# <center>**IROS**<center>

**Libraries**

In [1]:
import numpy as np
import mbloodmoon as bm
from iros_wrappers import perform_iros, compare_w_catalog
from iros_wrappers import gen_log, computes_params
from iros_wrappers import save_iros_output, load_iros_output
from iros_wrappers import save_iros_data, load_iros_data
from iros_wrappers import save_pickle, load_pickle

- Let's set up IROS

In [2]:
root_path = "/mnt/d/PhD_AASS/Coding/Images_fits/"
mask_file = root_path + "wfm_mask.fits"
simul_data = root_path + "iros_simulation_GC_LMC/20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb/"

cam_a = "cam1a"
cam_b = "cam1b"
dataset = "reconstructed"

filepaths = bm.simulation_files(simul_data)
wfm = bm.codedmask(mask_file, upscale_x=5, upscale_y=1)
sdlA = bm.simulation(filepaths[cam_a][dataset])
sdlB = bm.simulation(filepaths[cam_b][dataset])

max_iterations = 15
snr_threshold = 5

n_test = 0

- If IROS has been ran earlier and a FITS has been saved, this file will be loaded. If not, IROS will be launched.

In [3]:
iros_output_name = root_path + f"iros_output{n_test}.fits"

try:
    iros_output = load_iros_output(iros_output_name)
except FileNotFoundError:
    iros_output = perform_iros(
        wfm=wfm,
        sdlA=sdlA,
        sdlB=sdlB,
        cameras=(cam_a, cam_b),
        max_iterations=max_iterations,
        snr_threshold=snr_threshold,
    )
    save_iros_output(iros_output, mask_file, iros_output_name)

## Looping around the FOV...


15it [06:07, 24.47s/it]

# Saving data...
# Saving completed!


In [4]:
iros_output

{'cam1a': {'shiftx': array([  43.9241036 ,   13.55234697,  -28.13559855,   53.60453744,
           29.87281134,  -64.6       ,    8.44090305,   41.90002383,
           -6.24793145, -134.67055429,   42.9968241 ,  -41.75951971,
          -56.37006871,  174.9940355 ,  -56.39998558]),
  'shifty': array([  78.37877528,  -12.38659266,   29.15177642,  -26.77957834,
          -13.16681316,   39.00123457,   -1.94002471,  -24.54695805,
          -28.89931966,   82.32886906,   12.31392703,   36.15653489,
            6.4244692 , -106.54616734,   25.23177203]),
  'fluence': array([512698.88518584,  63210.85050942,  47467.67282463,  37951.14666424,
          30186.65655681,  21803.09959734,  19250.56352033,  16597.76875531,
          18493.21980649,  10832.79134786,  15936.41750458,  15747.05456817,
           9966.87319386,   5583.43829671,   8344.79035663]),
  'SNR': array([604.71447068,  91.59016557,  68.0173082 ,  56.52162511,
          50.63000319,  36.71480434,  29.60654797,  27.87024013,
    

- Let's take the output from the IROS loop and compute the useful parameters

In [5]:
iros_data_name = root_path + f"iros_data{n_test}.fits"

try:
    iros_data = load_iros_data(iros_data_name)

except FileNotFoundError:
    log = gen_log((cam_a, cam_b))

    iros_data = computes_params(
        iros_output=iros_output,
        wfm=wfm,
        sdlA=sdlA,
        sdlB=sdlB,
        log=log,
    )

    save_iros_data(
        data=iros_data,
        mask_file=mask_file,
        sdls=(sdlA, sdlB),
        save_to=iros_data_name,
    )

# Saving data...
# Saving completed!


 [astropy.io.fits.verify]


In [6]:
iros_data

{'cam1a': {'y': {'data': array([711, 485, 588, 449, 483, 613, 511, 454, 443, 721, 546, 606, 532,
          249, 579]),
   'format': 'J',
   'unit': 'px'},
  'x': {'data': array([5057, 4450, 3616, 5251, 4776, 2887, 4347, 5017, 4054, 1485, 5038,
          3343, 3051, 7678, 3051]),
   'format': 'J',
   'unit': 'px'},
  'shift_x': {'data': array([  43.9241036 ,   13.55234697,  -28.13559855,   53.60453744,
            29.87281134,  -64.6       ,    8.44090305,   41.90002383,
            -6.24793145, -134.67055429,   42.9968241 ,  -41.75951971,
           -56.37006871,  174.9940355 ,  -56.39998558]),
   'format': 'D',
   'unit': 'mm'},
  'dshift_x': {'data': array([0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025,
          0.025, 0.025, 0.025, 0.025, 0.025, 0.025]),
   'format': 'D',
   'unit': 'mm'},
  'shift_y': {'data': array([  78.37877528,  -12.38659266,   29.15177642,  -26.77957834,
           -13.16681316,   39.00123457,   -1.94002471,  -24.54695805,
           -28.89931

In [9]:
dataset_name = root_path + f"dataset_test{n_test}.fits"

try:
    dataset = load_iros_data(dataset_name)

except FileNotFoundError:
    catalogA = filepaths[cam_a]["sources"]
    catalogB = filepaths[cam_b]["sources"]

    dataset = compare_w_catalog(
        data=iros_data,
        catalogA=catalogA,
        catalogB=catalogB,
        cameras=(cam_a, cam_b),
        min_flux=0.1,
    )

    save_iros_data(
        data=dataset,
        mask_file=mask_file,
        sdls=(sdlA, sdlB),
        save_to=dataset_name,
    )


# Loading data...
# Loading completed!


In [10]:
dataset

{'cam1a': {'y': {'data': array([711, 485, 588, 449, 483, 613, 511, 454, 443, 721, 546, 606, 532,
          249, 579], dtype='>i4'),
   'format': 'J',
   'unit': 'px'},
  'x': {'data': array([5057, 4450, 3616, 5251, 4776, 2887, 4347, 5017, 4054, 1485, 5038,
          3343, 3051, 7678, 3051], dtype='>i4'),
   'format': 'J',
   'unit': 'px'},
  'shift_x': {'data': array([  43.9241036 ,   13.55234697,  -28.13559855,   53.60453744,
            29.87281134,  -64.6       ,    8.44090305,   41.90002383,
            -6.24793145, -134.67055429,   42.9968241 ,  -41.75951971,
           -56.37006871,  174.9940355 ,  -56.39998558], dtype='>f8'),
   'format': 'D',
   'unit': 'mm'},
  'dshift_x': {'data': array([0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025,
          0.025, 0.025, 0.025, 0.025, 0.025, 0.025], dtype='>f8'),
   'format': 'D',
   'unit': 'mm'},
  'shift_y': {'data': array([  78.37877528,  -12.38659266,   29.15177642,  -26.77957834,
           -13.16681316,   39.00123457

In [11]:
catalogA = bm.simulation(filepaths[cam_a]["sources"])
catalogB = bm.simulation(filepaths[cam_b]["sources"])

In [12]:
def check_pos(
    iros_data: dict,
    catalogs: tuple,
    sigma: int | float = 5,
    ):

    for cat, camera in zip(
        catalogs, iros_data.keys(),
    ):
        print(f"# CAMERA {camera.upper()}\n")
        for name, ra, dec, dra, ddec in zip(
            iros_data[camera]["catalog_name"]["data"],
            iros_data[camera]["ra"]["data"],
            iros_data[camera]["dec"]["data"],
            iros_data[camera]["dra"]["data"],
            iros_data[camera]["ddec"]["data"],
        ):
            cat_source_ra = cat.data[cat.data["NAME"] == name]["RA"][0]
            cat_source_dec = cat.data[cat.data["NAME"] == name]["DEC"][0]
            print(
                f"  - |IROS RA - {name.upper()} RA| < {sigma} * ddra: {np.abs(cat_source_ra - ra) < sigma*dra}\n"
                f"  - |IROS DEC - {name.upper()} DEC| < {sigma} * ddec: {np.abs(cat_source_dec - dec) < sigma*ddec}\n"
            )

# TODO: check how many sources are within chosen sigma wrt the ones which are not
check_pos(
    iros_data=dataset,
    catalogs=(catalogA, catalogB),
    sigma=5,
)

# CAMERA CAM1A

  - |IROS RA - SCOX1 RA| < 5 * ddra: True
  - |IROS DEC - SCOX1 DEC| < 5 * ddec: True

  - |IROS RA - GX5-1 RA| < 5 * ddra: True
  - |IROS DEC - GX5-1 DEC| < 5 * ddec: True

  - |IROS RA - GX349+2 RA| < 5 * ddra: True
  - |IROS DEC - GX349+2 DEC| < 5 * ddec: True

  - |IROS RA - GX17+2 RA| < 5 * ddra: True
  - |IROS DEC - GX17+2 DEC| < 5 * ddec: True

  - |IROS RA - GX9+1 RA| < 5 * ddra: True
  - |IROS DEC - GX9+1 DEC| < 5 * ddec: True

  - |IROS RA - GX340+0 RA| < 5 * ddra: True
  - |IROS DEC - GX340+0 DEC| < 5 * ddec: True

  - |IROS RA - GX3+1 RA| < 5 * ddra: True
  - |IROS DEC - GX3+1 DEC| < 5 * ddec: True

  - |IROS RA - GX13+1 RA| < 5 * ddra: True
  - |IROS DEC - GX13+1 DEC| < 5 * ddec: True

  - |IROS RA - X1820-303 RA| < 5 * ddra: True
  - |IROS DEC - X1820-303 DEC| < 5 * ddec: True

  - |IROS RA - CIRX1 RA| < 5 * ddra: True
  - |IROS DEC - CIRX1 DEC| < 5 * ddec: True

  - |IROS RA - GX9+9 RA| < 5 * ddra: True
  - |IROS DEC - GX9+9 DEC| < 5 * ddec: True

  - |IR

In [13]:
def check_counts(
    iros_data: dict,
    sdls: tuple,
    catalogs: tuple,
):

    for sdl, cat, camera in zip(
        sdls, catalogs, iros_data.keys(),
    ):
        print(f"# CAMERA {camera.upper()}\n")
        for name, f, of, sf, sp in zip(
            iros_data[camera]["catalog_name"]["data"],
            iros_data[camera]["fluence"]["data"],
            iros_data[camera]["obs_fluence"]["data"],
            iros_data[camera]["sub_fluence"]["data"],
            iros_data[camera]["simulphotons"]["data"],
        ):
            cat_source_ra = cat.data[cat.data["NAME"] == name]["RA"][0]
            cat_source_dec = cat.data[cat.data["NAME"] == name]["DEC"][0]
            # conv float64 to float32
            total_phs_simulated = len(
                sdl.data[(np.abs(sdl.data["RA"] - cat_source_ra) < 1e-7) & (np.abs(sdl.data["DEC"] - cat_source_dec) < 1e-7)]
            )
            print(
                f"  Counts stats for {name.upper()}:\n"
                f"    - Optimized wrt FITS: {f * 100 / total_phs_simulated:0.2f}%\n"
                f"    - Obs at peak wrt FITS: {of * 100 / total_phs_simulated:0.2f}%\n"
                f"    - Sub from sky wrt FITS: {sf * 100 / total_phs_simulated:0.2f}%\n"
                f"    - Retrieved wrt FITS: {sp * 100 / total_phs_simulated:0.2f}%\n"
            )


check_counts(
    iros_data=dataset,
    sdls=(sdlA, sdlB),
    catalogs=(catalogA, catalogB),
)

# CAMERA CAM1A

  Counts stats for SCOX1:
    - Optimized wrt FITS: 98.96%
    - Obs at peak wrt FITS: 92.97%
    - Sub from sky wrt FITS: 94.21%
    - Retrieved wrt FITS: 100.00%

  Counts stats for GX5-1:
    - Optimized wrt FITS: 102.64%
    - Obs at peak wrt FITS: 88.74%
    - Sub from sky wrt FITS: 88.82%
    - Retrieved wrt FITS: 100.00%

  Counts stats for GX349+2:
    - Optimized wrt FITS: 109.00%
    - Obs at peak wrt FITS: 94.67%
    - Sub from sky wrt FITS: 98.61%
    - Retrieved wrt FITS: 100.01%

  Counts stats for GX17+2:
    - Optimized wrt FITS: 107.43%
    - Obs at peak wrt FITS: 93.26%
    - Sub from sky wrt FITS: 93.85%
    - Retrieved wrt FITS: 100.00%

  Counts stats for GX9+1:
    - Optimized wrt FITS: 93.72%
    - Obs at peak wrt FITS: 91.04%
    - Sub from sky wrt FITS: 88.35%
    - Retrieved wrt FITS: 100.01%

  Counts stats for GX340+0:
    - Optimized wrt FITS: 102.05%
    - Obs at peak wrt FITS: 92.90%
    - Sub from sky wrt FITS: 89.22%
    - Retrieved wrt 